# AutoETF Research Agent MVP

面向行业 ETF 轮动的多因子研究、评分回测与权重诊断系统。

本 Notebook 只负责项目说明、调用 `src` 代码、展示结果和报告。正式运行前请先在 BigQuant 环境中执行字段探测。

In [ ]:
from src.agent_parser import parse_user_strategy
from src.factor_generator import generate_factor_candidates
from src.data_probe_bigquant import run_data_probe
from src.data_loader_bigquant import load_etf_data_bigquant
from src.factor_analysis import run_factor_analysis
from src.scoring import normalize_factors, build_weight_schemes, compute_composite_score
from src.backtest import run_top_pct_backtest
from src.diagnosis import diagnose_strategy
from src.report import generate_report, save_report
from src.plotting import plot_cumulative_return, plot_drawdown
from src.config import DEFAULT_CONFIG

config = DEFAULT_CONFIG

In [ ]:
def main():
    user_idea = "研究成交额放大、趋势走强、相对沪深300更强的行业ETF，未来是否更容易获得超额收益。"

    hypothesis = parse_user_strategy(user_idea)          # [AI-CORE]
    factors = generate_factor_candidates(hypothesis)     # [AI-CORE]

    # BigQuant 第一阶段必须先做字段探测。
    probe_results = run_data_probe()

    df = load_etf_data_bigquant(
        start_date=config.start_date,
        end_date=config.end_date,
        table_name=config.fund_table,
    )

    factor_cols = config.factor_cols
    target_cols = config.target_cols
    factor_directions = {factor["name"]: factor["direction"] for factor in factors}

    factor_result = run_factor_analysis(df, factor_cols, target_cols)
    weight_schemes = build_weight_schemes(factor_cols, factor_result)
    weights = weight_schemes["hypothesis_weight"]

    scored_df = normalize_factors(df, factor_directions)
    scored_df = compute_composite_score(scored_df, weights)

    backtest_result = run_top_pct_backtest(
        scored_df,
        score_col="composite_score",
        target_col="future_return_5d",
        top_pct=config.top_pct,
        min_holdings=config.min_holdings,
        cost_bps=config.cost_bps,
    )

    diagnosis = diagnose_strategy(factor_result, backtest_result, weights)  # [AI-CORE]
    report = generate_report(hypothesis, factors, factor_result, backtest_result, diagnosis, weights)  # [AI-CORE]
    save_report(report)

    plot_cumulative_return(backtest_result["daily_returns"]["return"])
    plot_drawdown(backtest_result["daily_returns"]["return"])

    return {
        "hypothesis": hypothesis,
        "factors": factors,
        "probe_tables": list(probe_results.keys()),
        "data_shape": df.shape,
        "weight_schemes": weight_schemes,
        "backtest_performance": backtest_result["performance"],
        "diagnosis": diagnosis,
        "report": report,
    }

result = main()
result